In [0]:
%sql
--Create Silver Table
CREATE TABLE IF NOT EXISTS silver.day19_customers (
    CustomerId INT,
    CustomerName STRING,
    City STRING,
    Age INT,
    UpdatedAt TIMESTAMP
)
USING DELTA;
--Create quarantine table
CREATE TABLE IF NOT EXISTS silver.day19_customer_quarantine (
    BatchId STRING,
    CustomerId INT,
    CustomerName STRING,
    City STRING,
    Age INT,
    UpdatedAt TIMESTAMP,
    ErrorReason STRING,
    RejectedAt TIMESTAMP,
    SourceFile STRING
)
USING DELTA;
--create data quality metrics
CREATE TABLE IF NOT EXISTS silver.day19_data_quality_metrics (
    BatchId STRING,
    TotalRecords INT,
    ValidRecords INT,
    InvalidRecords INT,
    QualityPercentage DOUBLE,
    ProcessedAt TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    TimestampType
)

customer_schema = StructType([
    StructField("CustomerId", IntegerType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("UpdatedAt", TimestampType(), True)
])

In [0]:
bronze_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(customer_schema)
    .load("/Volumes/workspace/bronze/day19/input/")
    .withColumn("SourceFile", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)


In [0]:
bronze_checkpoint = "/Volumes/workspace/bronze/day19/checkpoint/bronze_1"

bronze_query = (
    bronze_stream_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", bronze_checkpoint)
    .trigger(availableNow=True)
    .toTable("bronze.day19_customers")
)
bronze_query.awaitTermination()

In [0]:
%sql
SELECT *
FROM bronze.day19_customers
ORDER BY CustomerId;

CustomerId,CustomerName,City,Age,UpdatedAt,SourceFile,_ingested_at
201,Ravi,Chennai,28,2026-08-31T09:00:00.000Z,/Volumes/workspace/bronze/day19/input/Day19_Customer01.csv,2026-09-01T00:43:28.111Z
202,Arun,Bangalore,35,2026-08-31T09:01:00.000Z,/Volumes/workspace/bronze/day19/input/Day19_Customer01.csv,2026-09-01T00:43:28.111Z
203,null,Chennai,30,2026-08-31T09:02:00.000Z,/Volumes/workspace/bronze/day19/input/Day19_Customer01.csv,2026-09-01T00:43:28.111Z
204,Priya,null,25,2026-08-31T09:03:00.000Z,/Volumes/workspace/bronze/day19/input/Day19_Customer01.csv,2026-09-01T00:43:28.111Z
205,Kumar,Chennai,-5,2026-08-31T09:04:00.000Z,/Volumes/workspace/bronze/day19/input/Day19_Customer01.csv,2026-09-01T00:43:28.111Z
206,John,Chennai,150,2026-08-31T09:05:00.000Z,/Volumes/workspace/bronze/day19/input/Day19_Customer01.csv,2026-09-01T00:43:28.111Z
207,Meena,Madurai,29,2026-08-31T09:06:00.000Z,/Volumes/workspace/bronze/day19/input/Day19_Customer01.csv,2026-09-01T00:43:28.111Z
208,Anitha,Chennai,32,2026-08-31T09:07:00.000Z,/Volumes/workspace/bronze/day19/input/Day19_Customer01.csv,2026-09-01T00:43:28.111Z
208,Anitha,Chennai,32,2026-08-31T09:08:00.000Z,/Volumes/workspace/bronze/day19/input/Day19_Customer01.csv,2026-09-01T00:43:28.111Z
209,Suresh,Chennai,40,2026-08-31T10:00:00.000Z,/Volumes/workspace/bronze/day19/input/Day19_Customer02.csv,2026-09-01T00:43:28.111Z


In [0]:
silver_stream_df = (
    spark.readStream
    .table("bronze.day19_customers")
)

In [0]:
def validate_customers(df):

    duplicate_ids = (
        df.groupBy("CustomerId")
          .count()
          .filter(F.col("count") > 1)
          .select("CustomerId")
          .withColumn("IsDuplicate", F.lit(True))
    )

    validated_df = (
        df.join(
            duplicate_ids,
            on="CustomerId",
            how="left"
        )
        .fillna({"IsDuplicate": False})
        .withColumn(
            "ErrorReason",
            F.concat_ws(
                "; ",
                F.when(
                    F.col("CustomerId").isNull(),
                    "CustomerId is NULL"
                ),
                F.when(
                    F.col("CustomerName").isNull() |
                    (F.trim(F.col("CustomerName")) == ""),
                    "CustomerName is NULL"
                ),
                F.when(
                    F.col("City").isNull() |
                    (F.trim(F.col("City")) == ""),
                    "City is NULL"
                ),
                F.when(
                    F.col("Age").isNull() |
                    (F.col("Age") < 0) |
                    (F.col("Age") > 100),
                    "Age is outside valid range"
                ),
                F.when(
                    F.col("UpdatedAt").isNull(),
                    "UpdatedAt is NULL"
                ),
                F.when(
                    F.col("IsDuplicate") == True,
                    "Duplicate CustomerId"
                )
            )
        )
    )

    valid_df = (
        validated_df
        .filter(
            F.col("ErrorReason").isNull() |
            (F.col("ErrorReason") == "")
        )
        .drop("IsDuplicate", "ErrorReason")
    )

    invalid_df = (
        validated_df
        .filter(
            F.col("ErrorReason").isNotNull() &
            (F.col("ErrorReason") != "")
        )
        .drop("IsDuplicate")
        .withColumn(
            "RejectedAt",
            F.current_timestamp()
        )
    )

    return valid_df, invalid_df

In [0]:
from delta.tables import DeltaTable

def process_silver(batch_df, batch_id):

    print(f"Processing batch: {batch_id}")

    if batch_df.isEmpty():
        return
    
    batch_df=batch_df.drop( "_ingested_at")

    valid_df, invalid_df = validate_customers(batch_df)

    
    invalid_df=invalid_df.withColumn("BatchId", F.lit(batch_id ))
    invalid_df.write.mode("append").saveAsTable("silver.day19_customer_quarantine")

    silver_table = DeltaTable.forName(
        spark,
        "silver.day19_customers"
    )

    (
        silver_table.alias("target")
        .merge(
            valid_df.alias("source"),
            "target.CustomerId = source.CustomerId"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    total_count = batch_df.count()
    valid_count = valid_df.count()
    invalid_count = invalid_df.count()

    quality_percentage = (
        valid_count / total_count * 100
        if total_count > 0
        else 0
    )

    print(f"Batch ID        : {batch_id}")
    print(f"Total Records   : {total_count}")
    print(f"Valid Records   : {valid_count}")
    print(f"Invalid Records : {invalid_count}")
    print(f"Quality         : {quality_percentage:.2f}%")

    metrics_df = spark.createDataFrame(
        [
            (
                str(batch_id),
                total_count,
                valid_count,
                invalid_count,
                quality_percentage
            )
        ],
        [
            "BatchId",
            "TotalRecords",
            "ValidRecords",
            "InvalidRecords",
            "QualityPercentage"
        ]
    ).withColumn(
        "ProcessedAt",
        F.current_timestamp()
    )

    metrics_df.write.mode("append").saveAsTable("silver.day19_data_quality_metrics")

    QUALITY_THRESHOLD = 80.0
    if quality_percentage < QUALITY_THRESHOLD:
        raise Exception(f"Data quality check failed. Expected quality percentage >= {QUALITY_THRESHOLD}, found {quality_percentage}")

In [0]:
silver_checkpoint = "/Volumes/workspace/bronze/day19/checkpoint/silver"

silver_query = (
    silver_stream_df
    .writeStream
    .foreachBatch(process_silver)
    .option("checkpointLocation", silver_checkpoint)
    .trigger(availableNow=True)
    .start()
)

silver_query.awaitTermination()

26/09/01 01:18:08 Query 8235e5af-526c-478e-ad9f-4b48fe8e87f0 have exception: org.apache.spark.api.python.PythonException: [PYTHON_EXCEPTION] [root session: e69670b3-c92f-4dff-8556-61ae8c5294e4][cloned session: 0818ca58-7fcc-49d4-8a90-9a9ea6fa2c8e][userId: 75156637975438] Found error inside foreachBatch Python process: Traceback (most recent call last):
  File "/databricks/spark/python/_engine_pyspark.zip/_engine_pyspark/sql/connect/streaming/worker/foreach_batch_worker.py", line 172, in main
    process(df_ref_id, int(batch_id))
  File "/databricks/spark/python/_engine_pyspark.zip/_engine_pyspark/sql/connect/streaming/worker/foreach_batch_worker.py", line 76, in process
    func(batch_df, batch_id)
  File "/home/spark-19285cdc-0f53-4d9e-b8b7-82/.ipykernel/91/command-7528327981619797-3287636282", line 76, in process_silver
Exception: Data quality check failed. Expected quality percentage >= 80.0, found 0.0
 SQLSTATE: 38000
	at org.apache.spark.sql.connect.planner.StreamingForeachBatchHe

---------------------------------------------------------------------------
StreamingQueryException                   Traceback (most recent call last)
File <command-7528327981619798>, line 12
      1 silver_checkpoint = "/Volumes/workspace/bronze/day19/checkpoint/silver"
      3 silver_query = (
      4     silver_stream_df
      5     .writeStream
   (...)
      9     .start()
     10 )
---> 12 silver_query.awaitTermination()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/query.py:100, in StreamingQuery.awaitTermination(self, timeout)
     98 await_termination_cmd = pb2.StreamingQueryCommand.AwaitTerminationCommand()
     99 cmd.await_termination.CopyFrom(await_termination_cmd)
--> 100 self._execute_streaming_query_cmd(cmd)
    101 return None

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/query.py:188, in StreamingQuery._execute_streaming_query_cmd(self, cmd)
    186 exec_cmd = pb2.Command()
    187 exec_cmd.s

In [0]:
%sql
SELECT *
FROM silver.day19_customers
ORDER BY CustomerId;

In [0]:
%sql
SELECT
    BatchId,
    CustomerId,
    CustomerName,
    City,
    Age,
    ErrorReason,
    RejectedAt,
    SourceFile
FROM silver.day19_customer_quarantine
ORDER BY CustomerId;

BatchId,CustomerId,CustomerName,City,Age,ErrorReason,RejectedAt,SourceFile
0,203,null,Chennai,30,CustomerName is NULL,2026-09-01T01:01:54.832Z,/Volumes/workspace/bronze/day19/input/Day19_Customer01.csv
0,204,Priya,null,25,City is NULL,2026-09-01T01:01:54.832Z,/Volumes/workspace/bronze/day19/input/Day19_Customer01.csv
0,205,Kumar,Chennai,-5,Age is outside valid range,2026-09-01T01:01:54.832Z,/Volumes/workspace/bronze/day19/input/Day19_Customer01.csv
0,206,John,Chennai,150,Age is outside valid range,2026-09-01T01:01:54.832Z,/Volumes/workspace/bronze/day19/input/Day19_Customer01.csv
0,208,Anitha,Chennai,32,Duplicate CustomerId,2026-09-01T01:01:54.832Z,/Volumes/workspace/bronze/day19/input/Day19_Customer01.csv
0,208,Anitha,Chennai,32,Duplicate CustomerId,2026-09-01T01:01:54.832Z,/Volumes/workspace/bronze/day19/input/Day19_Customer01.csv
0,211,null,Chennai,31,CustomerName is NULL,2026-09-01T01:01:54.832Z,/Volumes/workspace/bronze/day19/input/Day19_Customer02.csv
0,212,Ramesh,Chennai,120,Age is outside valid range,2026-09-01T01:01:54.832Z,/Volumes/workspace/bronze/day19/input/Day19_Customer02.csv
2,217,null,Chennai,31,CustomerName is NULL,2026-09-01T01:17:58.650Z,/Volumes/workspace/bronze/day19/input/Day19_Customer04.csv
2,218,Vijay,Salem,130,Age is outside valid range,2026-09-01T01:17:58.650Z,/Volumes/workspace/bronze/day19/input/Day19_Customer04.csv


In [0]:
%sql
SELECT *
FROM silver.day19_data_quality_metrics
ORDER BY ProcessedAt DESC;

BatchId,TotalRecords,ValidRecords,InvalidRecords,QualityPercentage,ProcessedAt
2,2,0,2,0.0,2026-09-01T01:18:07.516Z
1,4,4,0,100.0,2026-09-01T01:08:37.693Z
0,13,5,8,38.46153846153847,2026-09-01T01:02:04.214Z


In [0]:
%sql
SELECT
    BatchId,
    ErrorReason,
    COUNT(*) AS RejectedRecords
FROM silver.day19_customer_quarantine
GROUP BY BatchId,ErrorReason
ORDER BY RejectedRecords DESC;